# Steps 4–6: Service Clients + Arena (`services/`)

**Goal:** Wrap each provider's SDK into one clean `async` function, then fire all three in parallel.

Files built in this session:
- `services/openai_client.py`
- `services/gemini_client.py`
- `services/claude_client.py`
- `services/arena.py`

---

## Why `async`?

When you call an API, your program sends a request and then *waits* for the response.
During that wait, a synchronous program sits completely idle — no other work happens.

With `async/await`, Python can *pause* a waiting function and run something else in the meantime.
This is what lets `arena.py` fire all three API calls **simultaneously** instead of one after another.

```
Synchronous (sequential):        Async (parallel):
  OpenAI  → wait 2s               OpenAI  ─┐
  Gemini  → wait 1.5s             Gemini  ─┤ all fire at once
  Claude  → wait 1s               Claude  ─┘
  Total:    4.5s                  Total:  ~2s (slowest wins)
```

---

## The shared contract

All three clients have the same function signature:

```python
async def chat(request: ChatRequest, model_id: str) -> ChatResponse:
```

One input shape, one output shape. `arena.py` calls all three without caring which SDK they use internally.
This is called the **adapter pattern** — each client adapts a different external API to our internal format.

---

## `openai_client.py`

In [ ]:
from typing import cast
from openai import AsyncOpenAI
from openai.types.chat import ChatCompletionMessageParam

# cast() is a typing tool — it tells the type checker "trust me, this is the type I say it is".
# At runtime it does nothing; it only exists to satisfy Pylance/mypy.
# We need it here because model_dump() returns dict[str, Any] but the OpenAI SDK
# expects the more specific ChatCompletionMessageParam type.

async def openai_chat_sketch(messages_raw, model_id, max_tokens):
    client = AsyncOpenAI(api_key="...")  # AsyncOpenAI, not OpenAI — the async version

    messages = cast(list[ChatCompletionMessageParam], messages_raw)

    response = await client.chat.completions.create(
        model=model_id,
        messages=messages,
        max_tokens=max_tokens,
    )

    # response.choices is a list — we always request one completion, so index 0
    content = response.choices[0].message.content or ""

    # response.usage can be None if the API doesn't return it (rare but possible)
    usage = response.usage
    prompt_tokens = usage.prompt_tokens if usage else 0
    completion_tokens = usage.completion_tokens if usage else 0

### Key decisions in `openai_client.py`:

**`AsyncOpenAI` not `OpenAI`**  
The openai package ships two clients: `OpenAI` (blocking) and `AsyncOpenAI` (async).
We must use the async one so `await` works and `asyncio.gather` can parallelise calls.

**`cast(list[ChatCompletionMessageParam], ...)`**  
`model_dump()` on a Pydantic model returns `dict[str, Any]` — a generic dict.
The OpenAI SDK's `messages` parameter expects a more specific typed dict (`ChatCompletionMessageParam`).
The shape is identical at runtime, but Pylance can't prove it statically.
`cast()` tells the type checker "these are the same thing" without any runtime cost.

**`usage.prompt_tokens if usage else 0`**  
The OpenAI SDK types `response.usage` as `Optional[CompletionUsage]` — it can theoretically be `None`.
In practice it's always present for chat completions, but we guard anyway.
If we wrote `usage.prompt_tokens` directly, Pylance would error on a potential `None` access.

**`choice.message.content or ""`**  
`content` is typed as `Optional[str]` — it's `None` when the model uses tool calls instead of text.
We don't use tools, so it's always a string, but the `or ""` satisfies the type checker cleanly.

---

## `gemini_client.py`

### Why we switched from `google-generativeai` to `google-genai`

During testing, the old `google-generativeai` package printed:

> `FutureWarning: All support for the google.generativeai package has ended.`

Google deprecated it in favour of `google-genai`. We updated `pyproject.toml` and rewrote the client.
The new SDK's async client lives at `client.aio.models.generate_content(...)`.

### Message format differences

Every provider has a slightly different message format. Gemini is the most different:

In [ ]:
from google.genai import types

# OpenAI / Claude format:
openai_messages = [
    {"role": "user",      "content": "Hello"},
    {"role": "assistant", "content": "Hi there!"},
]

# Gemini format — two differences:
#   1. "assistant" → "model"
#   2. content is a list of Parts, not a plain string
gemini_messages = [
    types.Content(role="user",  parts=[types.Part(text="Hello")]),
    types.Content(role="model", parts=[types.Part(text="Hi there!")]),
]

# Our helper _to_gemini_contents() handles this conversion automatically.

### Token fields — each provider uses different names

| Provider | Input tokens field | Output tokens field |
|---|---|---|
| OpenAI | `usage.prompt_tokens` | `usage.completion_tokens` |
| Gemini | `usage_metadata.prompt_token_count` | `usage_metadata.candidates_token_count` |
| Claude | `usage.input_tokens` | `usage.output_tokens` |

All three map to `prompt_tokens` and `completion_tokens` in our `ChatResponse`.

**`usage_metadata` can be `None`** — same reasoning as OpenAI's `usage`.
We guard with `usage.prompt_token_count if usage else 0`.

---

## `claude_client.py`

Claude's SDK is the cleanest of the three — `AsyncAnthropic` works identically to `AsyncOpenAI`.
The main difference is how you access the response content.

In [ ]:
from anthropic.types import TextBlock

# Claude's response.content is a list of blocks, not a plain string.
# Blocks can be: TextBlock, ThinkingBlock, ToolUseBlock, and others.
# We only care about text, so we find the first TextBlock explicitly.

# WRONG — Pylance error: not all blocks have .text
# content = response.content[0].text

# CORRECT — narrow to TextBlock first
# text = next(block.text for block in response.content if isinstance(block, TextBlock))

# isinstance() narrows the type — inside the generator, Pylance knows block is a TextBlock
# and that .text is safe to access.

### Why does Claude return a list of blocks?

Claude supports **extended thinking** (reasoning tokens before answering) and **tool use**.
When those are active, the response contains multiple blocks — a `ThinkingBlock` followed by a `TextBlock`, 
or a `ToolUseBlock` alongside text.

We're not using those features, so there's always exactly one `TextBlock` — but the SDK types 
the list as a union of all possible block types. Filtering with `isinstance` is the correct approach.

---

## Error handling — same pattern in all three clients

All three clients wrap the API call in `try/except Exception`:

In [ ]:
# Illustrative pattern — same in all three clients
try:
    # ... API call ...
    return ChatResponse(provider=..., content="...", ...)  # success

except Exception as e:
    return ChatResponse(provider=..., content="", error=str(e))  # failure

**Why catch everything?** — API calls can fail in many ways: network timeout, invalid key,
rate limit (429), model overloaded (503), content filtered, etc. We catch them all and return
a `ChatResponse` with `error` set instead of propagating an exception.

**Why is this important for the UI?** — If one provider fails, the other two should still show
their results. Propagating an exception from one client would crash `asyncio.gather` and wipe
all three responses. By returning an error-flagged `ChatResponse`, the UI can display the error
in that column while the other two columns show results normally.

---

## `arena.py` — the orchestrator

In [ ]:
import asyncio

# asyncio.gather fires all coroutines at the same time and waits for all to finish.
# The results come back in the same order as the arguments — not arrival order.

# openai_response, gemini_response, claude_response = await asyncio.gather(
#     openai_client.chat(request, openai_model),   # starts immediately
#     gemini_client.chat(request, gemini_model),   # starts immediately
#     claude_client.chat(request, claude_model),   # starts immediately
# )

# All three are in-flight at the same time.
# gather() returns when the last one finishes.
# Total wait ≈ slowest provider, not sum of all three.

### Why `asyncio.gather` and not `asyncio.wait` or threads?

**`asyncio.gather`** — simplest, returns results in input order, re-raises exceptions. Perfect for
"fire N coroutines, get N results back in order."

**`asyncio.wait`** — more control (e.g. return as tasks complete), but more boilerplate. Not needed here.

**Threads (`ThreadPoolExecutor`)** — works for blocking I/O but adds overhead and complexity.
Since all three SDKs have proper async clients, `asyncio.gather` is the right tool.

---

## Live test output

After fixing all type errors and upgrading the Gemini SDK, the test run produced:

```
[openai] GPT-4.1 Nano    → ERROR: 429 insufficient_quota  (billing not set up)
[gemini] Gemini 2.0 Flash Lite → ERROR: 429 RESOURCE_EXHAUSTED  (free tier daily limit)
[claude] Claude Haiku 4.5
  Content:  Hello! It's nice to meet you.
  Tokens:   13 in / 12 out
  Latency:  0.78s
  Cost:     $0.000058
```

The 429s are account-level quota issues, not code bugs.
Error handling worked correctly — one provider failing did not affect the other two.

**To fix:**
- **OpenAI** → Add billing at platform.openai.com → Billing
- **Gemini** → Free tier resets daily; or switch to a paid model (`gemini-2.5-flash`)

---

## Summary

| What | Why |
|---|---|
| All clients share the same `async def chat(request, model_id) -> ChatResponse` | Uniform interface — `arena.py` doesn't know or care which SDK it's calling |
| `cast()` for message lists | SDK types expect specific TypedDicts; `cast` bridges the gap without runtime cost |
| `usage if usage else 0` | SDK types `usage` as Optional — guard prevents None attribute access |
| `isinstance(block, TextBlock)` | Claude returns a union of block types; narrowing is required to safely access `.text` |
| `try/except` returns `ChatResponse(error=...)` | One provider failing must not crash the other two |
| `asyncio.gather` in `arena.py` | Fires all three in parallel; total latency ≈ slowest, not sum |
| Switched to `google-genai` | `google-generativeai` was officially deprecated during development |

**Next step:** Session 7 — terminal smoke test via `__init__.py` main().